# 03 — Comparison and the pre-registered decision

Paired match bootstrap (1000 draws, seed 20261108, 95 %). H1: `exp4_full_ens3` − `exp3_ref_ens3`, `target_like`; provider CI lower bound > 0 **and** `L_ctrl` CI lower bound > −0.01, each with its own nested re-selection. Everything else is report-only.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp4_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_4')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp4_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp4_protocol as P, exp4_report as R
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PREREG verified 254e2e216cb05154 | frozen 2026-11-09T17:09:15+00:00


In [2]:
reference = pd.read_csv(C.C3.RESULTS / 'reference_rows.csv')
reference = reference[reference.label_set.isin(C.S1_LABELS)]
rows = R.collect_rows([reference, pd.read_csv(C.RESULTS / 'argmax_rows.csv')])
rows.to_csv(C.RESULTS / 'per_match_results.csv', index=False)
table = R.summary_table(rows)
table.to_csv(C.RESULTS / 'comparison_table.csv', index=False)
cols = ['variant', 'mean_spell_f1', 'min_spell_f1', 'mean_macro_f1', 'mean_count_error', 'max_abs_count_error',
        'delta_vs_ref_ens3', 'ci95_low_vs_ref', 'ci95_high_vs_ref', 'wins_vs_ref', 'count_gate']
for label_set in C.S1_LABELS:
    for view in C.VIEWS:
        part = table[(table.label_set == label_set) & (table.view == view)]
        print(f'== {label_set} / {view}'); print(R.as_markdown(part, cols)); print()

== provider / original
| variant | mean_spell_f1 | min_spell_f1 | mean_macro_f1 | mean_count_error | max_abs_count_error | delta_vs_ref_ens3 | ci95_low_vs_ref | ci95_high_vs_ref | wins_vs_ref | count_gate |
|---|---|---|---|---|---|---|---|---|---|---|
| exp4_full_ens3 | 0.674 | 0.618 | 0.923 | -0.220 | 0.341 | +0.007 | -0.007 | +0.026 | 3 | False |
| exp4_touch_ens3 | 0.681 | 0.628 | 0.924 | -0.218 | 0.348 | +0.015 | +0.006 | +0.025 | 6 | False |
| exp4_aux_ens3 | 0.667 | 0.610 | 0.922 | -0.215 | 0.360 | +0.001 | -0.005 | +0.007 | 4 | False |
| exp3_ref_ens3 | 0.666 | 0.621 | 0.922 | -0.204 | 0.360 | +0.000 | +0.000 | +0.000 | 0 | False |
| exp2_aug | 0.657 | 0.596 | 0.917 | -0.200 | 0.354 | -0.009 | -0.016 | -0.002 | 2 | False |
| exp4_full | 0.662 | 0.622 | 0.920 | -0.203 | 0.303 | -0.005 | -0.015 | +0.006 | 4 | False |
| exp4_full_s1 | 0.652 | 0.608 | 0.917 | -0.221 | 0.344 | -0.014 | -0.024 | -0.004 | 2 | False |
| exp4_full_s2 | 0.665 | 0.625 | 0.921 | -0.197 | 0.308 | -0.001 | -

## Per match, `target_like` (spellF1, count error in brackets)

In [3]:
show = ('exp4_full_ens3', 'exp3_ref_ens3', 'exp4_touch_ens3', 'exp4_aux_ens3', 'exp2_aug')
for label_set in C.DECISION_LABELS:
    print(f'== {label_set}'); print(R.per_match(rows, label_set, 'target_like', show).to_string()); print()

== provider
variant exp4_full_ens3  exp3_ref_ens3 exp4_touch_ens3  exp4_aux_ens3       exp2_aug
match                                                                              
J03WMX   0.615 (-0.34)  0.615 (-0.37)   0.617 (-0.36)  0.631 (-0.33)  0.621 (-0.35)
J03WN1    0.64 (-0.26)  0.659 (-0.29)   0.665 (-0.24)  0.656 (-0.28)  0.639 (-0.24)
J03WOH   0.684 (-0.29)   0.67 (-0.23)   0.683 (-0.24)  0.665 (-0.23)  0.668 (-0.21)
J03WOY   0.702 (-0.18)  0.695 (-0.16)   0.707 (-0.19)  0.699 (-0.21)  0.696 (-0.14)
J03WPY   0.684 (-0.19)  0.691 (-0.18)   0.701 (-0.21)  0.703 (-0.19)  0.684 (-0.18)
J03WQQ   0.723 (-0.06)  0.698 (-0.06)   0.711 (-0.12)  0.709 (-0.07)  0.685 (-0.07)
J03WR9   0.654 (-0.19)  0.621 (-0.13)   0.671 (-0.15)   0.61 (-0.16)  0.604 (-0.19)

== ctrl
variant exp4_full_ens3  exp3_ref_ens3 exp4_touch_ens3  exp4_aux_ens3       exp2_aug
match                                                                              
J03WMX   0.729 (-0.15)  0.748 (-0.14)   0.728 (-0.14)  

## exp_0 quality gates (vs the same view's baseline), provider and `L_ctrl`

In [4]:
gates = R.gate_rows(rows, 'provider') + R.gate_rows(rows, 'ctrl')
print(pd.DataFrame([{'label_set': g['label_set'], 'view': g['view'], 'variant': g['variant'], **g['checks'], 'pass': g['pass'],
                     'worst_count_error': round(g['worst_count_error'], 3), 'worst_match': g['worst_count_match']}
                    for g in gates]).to_string(index=False))

label_set        view         variant  gain_at_least_005  ci_lower_above_zero  macro_regression_at_most_001  all_match_counts_within_30pct  pass  worst_count_error worst_match
 provider    original  exp4_full_ens3               True                 True                          True                          False False              0.341      J03WMX
 provider    original exp4_touch_ens3               True                 True                          True                          False False              0.348      J03WMX
 provider    original   exp4_aux_ens3               True                 True                          True                          False False              0.360      J03WMX
 provider    original   exp3_ref_ens3               True                 True                          True                          False False              0.360      J03WMX
 provider    original        exp2_aug               True                 True                          True             

## Pre-registered decision (H1), seed honesty, adoption tree

In [5]:
decision = R.decide(rows)
C1.write_json(C.RESULTS / 'decision.json', decision)
for name, rule in decision['rules'].items():
    print(f"{name:26s} {rule['mean_delta']:+.4f} [{rule['ci95'][0]:+.4f}, {rule['ci95'][1]:+.4f}] wins {rule['wins']}/7 -> {'pass' if rule['passed'] else 'FAIL'}")
print('H1', 'PASSED' if decision['H1']['passed'] else 'FAILED', '|', decision['seed_honesty']['statement'],
      '| seed spread', decision['seed_honesty']['exp4_full_seed_spread_provider_target_like'])
print('adoption:', json.dumps(decision['adoption']))
ro = decision['report_only']
for item in ro['H1_original_view'] + ro['ablations_vs_reference'] + ro['full_vs_ablations'] + ro['S1_full_vs_reference'] + ro['single_seeds_vs_reference']:
    print(f"  {item['label_set']:8s} {item['view']:12s} {item['a']:16s} - {item['b']:16s} {item['mean_delta']:+.4f} [{item['ci95'][0]:+.4f}, {item['ci95'][1]:+.4f}] wins {item['wins']}")
print(pd.DataFrame([{k: v for k, v in s.items() if k != 'seeds'} for s in ro['seed_spreads']]).round(4).to_string(index=False))

H1_provider                +0.0074 [-0.0053, +0.0193] wins 4/7 -> FAIL
H1_ctrl_non_inferiority    -0.0062 [-0.0160, +0.0027] wins 3/7 -> FAIL
H1 FAILED | Δ is smaller than the seed spread: called noise in REPORT.md | seed spread 0.016682818302346747
adoption: {"write_files": false, "statement": "H1 failed: no exp_4 files are written; exp_3's two BAR-ATH files stay the team's options"}
  provider original     exp4_full_ens3   - exp3_ref_ens3    +0.0074 [-0.0074, +0.0256] wins 3
  ctrl     original     exp4_full_ens3   - exp3_ref_ens3    -0.0040 [-0.0162, +0.0071] wins 4
  provider original     exp4_touch_ens3  - exp3_ref_ens3    +0.0146 [+0.0061, +0.0249] wins 6
  provider target_like  exp4_touch_ens3  - exp3_ref_ens3    +0.0151 [+0.0069, +0.0274] wins 7
  ctrl     original     exp4_touch_ens3  - exp3_ref_ens3    -0.0032 [-0.0176, +0.0111] wins 3
  ctrl     target_like  exp4_touch_ens3  - exp3_ref_ens3    -0.0032 [-0.0200, +0.0131] wins 3
  provider original     exp4_aux_ens3    - exp3_

## Generalisation gap and short-possession recall (report only)

In [6]:
gap = pd.read_csv(C.RESULTS / 'gap_table.csv')
print(gap[gap.level == 'family'].round(4).to_string(index=False))
recall = pd.read_csv(C.RESULTS / 'short_recall.csv')
recall_summary = recall.groupby(['view', 'variant'])[['recall_short_decoded', 'recall_short_argmax', 'recall_all_decoded']].mean()
print(recall_summary.round(4).to_string())
report = {'prereg_sha256': protocol['prereg_sha256'], 'bootstrap': 'paired match bootstrap, 1000 draws, seed 20261108, 95% percentile',
          'table': R.records(table), 'gates': gates, 'decision': decision,
          'gap_table': R.records(gap), 'short_recall': R.records(recall_summary.reset_index()),
          'repro_check': C1.read_json(C.RESULTS / 'repro_check.json')}
C1.write_json(C.RESULTS / 'proxy_report.json', report)

        arm  train_nll  heldout_nll_original  heldout_nll_target_like  gap_original  gap_target_like family  gap_growth_vs_exp2_recipe  level flag_gap_growth_gt_25pct
exp2_recipe     0.1165                0.1884                   0.1913        0.0719           0.0747    NaN                     0.0000 family                    False
   exp4_aux     0.1181                0.1877                   0.1903        0.0696           0.0722    NaN                    -0.0322 family                    False
  exp4_full     0.1246                0.1770                   0.1815        0.0524           0.0569    NaN                    -0.2718 family                    False
 exp4_touch     0.1237                0.1764                   0.1818        0.0527           0.0581    NaN                    -0.2664 family                    False
                             recall_short_decoded  recall_short_argmax  recall_all_decoded
view        variant                                                       

## Figure (proxy only)

In [7]:
order = ['exp3_ref_ens3', 'exp4_touch_ens3', 'exp4_aux_ens3', 'exp4_full_ens3']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label_set, marker in zip(C.S1_LABELS, ('o', 's', '^', 'v')):
    part = table[(table.label_set == label_set) & (table.view == 'target_like')].set_index('variant').reindex(order)
    axes[0].plot(order, part.mean_spell_f1, marker=marker, label=label_set)
    axes[1].plot(order, part.delta_vs_ref_ens3, marker=marker, label=label_set)
axes[0].set_ylabel('mean spellF1, target_like (7 held-out matches)'); axes[0].set_title('spellF1 by arm and label set')
axes[1].axhline(0, color='0.5', lw=0.8); axes[1].set_ylabel('Δ vs exp3_ref_ens3'); axes[1].set_title('paired Δ (point estimate)')
for ax in axes:
    ax.tick_params(axis='x', rotation=25); ax.legend(title='labels')
fig.tight_layout(); fig.savefig(C.RESULTS / 'fig_exp4.png', dpi=130)
print('saved', C.RESULTS / 'fig_exp4.png')

saved /home/jeremy-mboe/Documents/Kuliah/Sem5/barca/exp_jeremy/exp_4/results/fig_exp4.png
